# Order Payments - Silver Transformation

## Setup

In [0]:
from pyspark.sql.functions import col, trim

environment = "dev"

catalog = f"ecommerce_{environment}"
source_table = f"{catalog}.bronze.olist_order_payments"
target_table = f"{catalog}.silver.olist_order_payments"

## Read Bronze data

In [0]:
bronze_df = spark.table(source_table)

## Review the data

In [0]:
bronze_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: decimal(8,2) (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(bronze_df.limit(10))

order_id,payment_sequential,payment_type,payment_installments,payment_value,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments


In [0]:
print("Number of rows:", bronze_df.count())
print("Number of columns:", len(bronze_df.columns))

Number of rows: 103886
Number of columns: 12


In [0]:
rescued_row_count = (
    bronze_df
    .filter(col("_rescued_data").isNotNull())
    .filter(trim(col("_rescued_data")) != "")
    .count()
)

print("Number of rescued rows:", rescued_row_count)

Number of rescued rows: 0


In [0]:
for column, dtype in bronze_df.dtypes[:5]:
    print(column)
    print("Null count:", bronze_df.filter(col(column).isNull()).count())
    print("Distinct count:", bronze_df.filter(col(column).isNotNull()).select(column).distinct().count())

    if dtype == "string":
        print("Extra whitespace row count:",
            (
                bronze_df.withColumn(f"{column}_trimmed", trim(col(column)))
                .filter(col(column) != col(f"{column}_trimmed"))
                .count()
            )
        )
    print("-"*20)

order_id
Null count: 0
Distinct count: 99440
Extra whitespace row count: 0
--------------------
payment_sequential
Null count: 0
Distinct count: 29
--------------------
payment_type
Null count: 0
Distinct count: 5
Extra whitespace row count: 0
--------------------
payment_installments
Null count: 0
Distinct count: 24
--------------------
payment_value
Null count: 0
Distinct count: 29077
--------------------


In [0]:
bronze_df.count() == bronze_df.select("order_id","payment_sequential").distinct().count()

True

The combination of order_id and payment_sequential is unique and identifies each payment record.

## Transform to Silver

In [0]:
silver_df = (
    bronze_df
    .withColumnRenamed(
        "payment_sequential",
        "payment_sequence_number"
    )
    .withColumnRenamed(
        "payment_installments",
        "payment_installment_count"
    )
    .withColumnRenamed(
        "payment_value",
        "payment_amount"
    )
)

## Write to Silver

In [0]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

## Review the result

In [0]:
silver_table_df = spark.table(target_table)

silver_table_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- payment_sequence_number: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installment_count: integer (nullable = true)
 |-- payment_amount: decimal(8,2) (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(silver_table_df.limit(10))

order_id,payment_sequence_number,payment_type,payment_installment_count,payment_amount,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95,null,/Volumes/ecommerce_dev/landing/raw_files/olist/order_payments/olist_order_payments_dataset.csv,2026-08-02T21:29:10.000Z,2026-08-02T22:44:00.510Z,8e58f346-fe1b-45d0-9cbb-d96fd20e0a7b,olist,order_payments


In [0]:
print("Bronze row count:", bronze_df.count())
print("Silver row count:", silver_table_df.count())

Bronze row count: 103886
Silver row count: 103886
